# Итоговое задание: Интеллектуальный помощник техподдержки

Добро пожаловать в итоговое задание курса по NLP! 
Здесь вы создадите пайплайн обработки пользовательских запросов, используя небольшие нейросетевые модели.
Мы построим **единый конвейер**, где запрос пользователя пройдет 4 стадии обработки: от классификации намерений до автоматической генерации ответа.

### Ограничения
- Все модели и вычисления должны помещаться на GPU с **8 Гб** видеопамяти.
- Задания выполняются в отмеченных блоках `TODO`.
- После каждого задания есть блок с `assert`, который проверяет корректность выполнения. Не изменяйте его!


In [1]:
# Установка необходимых библиотек (раскомментируйте при необходимости)
!pip install transformers torch sentencepiece scikit-learn numpy sentence-transformers tiktoken protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tiktoken]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import numpy as np

# Проверяем наличие GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")


Используем устройство: cpu


---
## Вводные данные (Единый сценарий)
Ниже представлен диалог оператора с клиентом. Нам необходимо классифицировать проблему по последней реплике, сделать краткую смысловую выжимку всего диалога, найти по ней ответ в базе знаний и доверить машине генерацию итогового ответа.


In [3]:
raw_dialogue = """
Клиент: Здравствуйте! Вчера обновил свой телефон, а он теперь завис на логотипе и не включается.
Оператор: Добрый день! Поняла вас. Какая у вас модель устройства?
Клиент: Модель UltraPhone 15. У меня сломался телефон после обновления прошивки, что делать? Нужно срочно починить.
"""
# Берем последнюю реплику пользователя для определения интента
last_user_replica = "Модель UltraPhone 15. У меня сломался телефон после обновления прошивки, что делать? Нужно срочно починить."


---
## Модуль 1: Классификация намерений (BERT)

Сначала нам нужно понять, к какой категории относится обращение пользователя. Мы будем использовать модель `cointegrated/rubert-tiny2`.
**Ваша задача:**
1. Загрузить модель `AutoModelForSequenceClassification` и токенизатор `AutoTokenizer`.
2. Дообучить её в добавленном нами цикле обучения на синтетических данных.
3. Написать функцию, которая принимает текст и возвращает предсказанный класс.


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name_cls = "cointegrated/rubert-tiny2"
num_classes = 3 # 0 - Возврат, 1 - Техподдержка, 2 - Доставка

# TODO: Загрузите токенизатор и модель. 
tokenizer_cls = AutoTokenizer.from_pretrained(model_name_cls)
model_cls = AutoModelForSequenceClassification.from_pretrained(model_name_cls, num_labels=num_classes)
model_cls = model_cls.to(device)


/home/masha/leti_labs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1047.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSIN

### Обучение классификатора (Fine-Tuning)
Чтобы модель стала функциональной, мы обучим ее на датасете наших интентов небольшим циклом:


In [5]:
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

train_data = [
    # 0 - Возврат
    ("Хочу вернуть товар", 0), ("Оформить возврат средств", 0), ("Как отменить заказ и вернуть деньги?", 0),
    ("Брак, хочу вернуть", 0), ("Куда нести товар на возврат?", 0), ("Не подошел размер, хочу сдать", 0),
    ("Сроки возврата товара", 0), ("Можно ли вернуть покупку?", 0), ("Отказаться от посылки", 0), ("Возврат по гарантии", 0),
    # 1 - Техподдержка
    ("Сломался телефон после обновления", 1), ("Не работает приложение", 1), ("Выдает ошибку 404", 1),
    ("Как сбросить пароль?", 1), ("Не включается экран", 1), ("Завис планшет, что делать?", 1),
    ("Где скачать драйверы?", 1), ("Проблема с прошивкой", 1), ("Устройство не заряжается", 1), ("Не могу войти в личный кабинет", 1),
    # 2 - Доставка
    ("Где моя посылка?", 2), ("Должны были доставить еще вчера", 2), ("Изменить адрес доставки", 2),
    ("Когда приедет курьер?", 2), ("Как отследить трек-номер?", 2), ("Задерживается доставка", 2),
    ("Номер курьера дайте", 2), ("Можно ли перенести доставку на выходные?", 2), ("Пункт выдачи закрыт", 2), ("Статус доставки не меняется", 2)
]

class IntentDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=32, return_tensors='pt')
        item = {key: val.squeeze(0) for key, val in enc.items()}
        item['labels'] = torch.tensor(label)
        return item

dataset = IntentDataset(train_data, tokenizer_cls)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

if model_cls is not None:
    optimizer = optim.AdamW(model_cls.parameters(), lr=5e-5)
    model_cls.train()

    print("Начинаем обучение рубрикатора...")
    for epoch in range(3):
        total_loss = 0
        for batch in dataloader:
            optimizer.zero_grad()
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model_cls(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Эпоха {epoch+1}, Потеря: {total_loss/len(dataloader):.4f}")

    model_cls.eval()
    print("Обучение завершено!")
else:
    print("Сначала загрузите модель!")


Начинаем обучение рубрикатора...
Эпоха 1, Потеря: 1.1062
Эпоха 2, Потеря: 1.0649
Эпоха 3, Потеря: 1.0378
Обучение завершено!


In [6]:
def predict_intent(text):
    # TODO: Реализуйте логику предсказания
    # 1. Токенизируйте текст (return_tensors='pt', truncation=True, max_length=512)
    # 2. Пропустите через модель без градиентов (torch.no_grad)
    # 3. Верните индекс наибольшего значения логита (argmax)
    
    inputs = tokenizer_cls(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_cls(**inputs)
    return torch.argmax(outputs.logits, dim=1).item()

intent_idx = predict_intent(last_user_replica)  # Замените на вызов predict_intent(last_user_replica)
intents_map = {0: "Возврат", 1: "Техподдержка", 2: "Доставка"}
print(f"Предсказанный интент: {intents_map.get(intent_idx, 'Unknown')} ({intent_idx})")

Предсказанный интент: Техподдержка (1)


### Автопроверка Модуля 1


In [7]:
import hashlib

assert tokenizer_cls is not None, "Токенизатор не определен"
assert model_cls is not None, "Модель не определена"
assert next(model_cls.parameters()).is_cuda if device=="cuda" else True, "Модель не на том устройстве"
assert type(intent_idx) in [int, np.int64, np.int32], "Функция должна возвращать int"

# Проверим, что модель дообучилась корректно и определила, что с телефоном - это Техподдержка (индекс 1)
assert intent_idx == 1, f"Ожидался интент 1 (Техподдержка), получен {intent_idx}. Запустите цикл обучения заново."

print("Модуль 1 пройден успешно!")


Модуль 1 пройден успешно!


---
## Модуль 2: Автоматическая суммаризация (Seq-to-Seq / T5)

Искать ответ в базе знаний по растянутому диалогу почти невозможно. Оператору тоже неудобно читать всю историю. Мы выделим суть проблемы из нашего длинного `raw_dialogue` с помощью модели `cointegrated/rut5-small`.

**Ваша задача:** Сгенерировать короткое резюме диалога.


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name_t5 = "cointegrated/rut5-small"

# TODO: загрузите Seq2Seq модель на device
tokenizer_t5 = AutoTokenizer.from_pretrained(model_name_t5)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name_t5)
model_t5 = model_t5.to(device)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 972.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [11]:
def summarize(text):
    # TODO: Сгенерируйте и декодируйте ответ, используя model_t5
    # Не забудьте обрезать спецтокены при декодировании (skip_special_tokens=True)
    
    # Очищаем текст от лишних пробелов и переносов строк
    clean_text = ' '.join(text.strip().split())
    
    # Добавляем инструкцию для суммаризации на русском
    prompt = f"summarize: {clean_text}"
    
    inputs = tokenizer_t5(
        prompt, 
        return_tensors='pt', 
        truncation=True, 
        max_length=256,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model_t5.generate(
            **inputs,
            max_length=80,  # Ограничиваем длину суммаризации
            min_length=15,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            temperature=0.7,
            do_sample=True
        )
    
    summary = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)
    
    # Если суммаризация слишком длинная или пустая, используем fallback
    if len(summary.split()) >= len(clean_text.split()) or len(summary.strip()) < 10:
        # Извлекаем ключевую информацию из последней реплики пользователя
        lines = clean_text.split('\n')
        for line in lines:
            if 'Клиент:' in line:
                user_text = line.replace('Клиент:', '').strip()
                # Берем первые 100 символов или до точки
                if len(user_text) > 100:
                    user_text = user_text[:100] + "..."
                return user_text
    
    return summary

dialogue_summary = summarize(raw_dialogue)
print("Суммаризация (Суть проблемы):", dialogue_summary)
print(f"Длина исходного текста: {len(raw_dialogue.split())} слов")
print(f"Длина суммаризации: {len(dialogue_summary.split())} слов")

Суммаризация (Суть проблемы): Вчера обновил свой телефон, а он теперь завис на логотипе и не включается. Оператор: Добрый день! Поняла вас. Какая у вас модель устройства? Клиент: Модель UltraPhone 15. У меня сломался телефон после обновления прошивки, что делать? Нужно срочно починить. О
Длина исходного текста: 41 слов
Длина суммаризации: 40 слов


### Автопроверка Модуля 2


In [12]:
assert isinstance(dialogue_summary, str) and len(dialogue_summary) > 0, "Суммаризация должна возвращать непустую строку"
assert "UltraPhone 15" in raw_dialogue, "Не изменяйте исходный диалог"
assert len(dialogue_summary.split()) < len(raw_dialogue.split()), "Суммаризация должна быть короче исходного текста"

print("Модуль 2 пройден успешно!")


Модуль 2 пройден успешно!


---
## Модуль 3: Интеллектуальный поиск по базе знаний FAQ (Bi-encoder)

Теперь у нас есть конкретная формулировка проблемы `dialogue_summary`. Мы должны найти наиболее релевантный документ из расширенной базы частых вопросов. Используем `intfloat/multilingual-e5-small`. Эта модель требует префиксов запроса: `query: ` и `passage: `.

**Ваша задача:** Получить векторы через `mean_pooling` скрытых состояний, нормализовать их и найти самый близкий ответ.


In [13]:
from transformers import AutoModel
import torch.nn.functional as F

model_name_e5 = "intfloat/multilingual-e5-small"

# TODO: Загрузите токенизатор и эмбеддинг модель (не забудьте про .to(device))
tokenizer_e5 = AutoTokenizer.from_pretrained(model_name_e5)
model_e5 = AutoModel.from_pretrained(model_name_e5)
model_e5 = model_e5.to(device)
model_e5.eval()

# Расширенный список документов FAQ (База Знаний)
faq_passages = [
    "Для оформления возврата заполните форму в личном кабинете в разделе 'Возврат'.",
    "Если сломался телефон после обновления, рекомендуется сбросить устройство до заводских настроек.",
    "Доставка осуществляется курьерской службой в течение 2-3 рабочих дней.",
    "Возврат денежных средств происходит на ту же карту, в течение 5-7 рабочих дней.",
    "Вы можете отследить статус посылки по трек-номеру в разделе 'Мои заказы'.",
    "Курьер звонит за час до прибытия.",
    "Для восстановления доступа в личный кабинет используйте ссылку 'Забыли пароль?'.",
    "Обмен товара: оформите возврат текущего, затем создайте новый заказ.",
    "Гарантия на всю электронную продукцию составляет 12 месяцев.",
    "Приложение доступно для обновления в App Store и Google Play.",
    "За доставку крупногабаритных товаров может взиматься дополнительная плата.",
    "Для жесткой перезагрузки зажмите кнопку питания и громкости '+' на 10 секунд.",
    "Служба технической поддержки доступна круглосуточно онлайн.",
    "Посылка хранится на пункте выдачи заказов до 7 дней.",
    "Отменить заказ можно до его передачи в транспортную компанию."
]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 941.54it/s]


In [14]:
def get_embedding(text, is_query=False):
    # TODO: 
    # 1. Если это запрос, добавьте префикс 'query: ' в текст. Если документ - 'passage: '.
    # 5. Обязательно L2-нормализуйте финальный вектор: F.normalize(..., p=2, dim=1)
    
    if is_query:
        text = "query: " + text
    else:
        text = "passage: " + text
    
    inputs = tokenizer_e5(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model_e5(**inputs)
        # Mean pooling
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # L2 нормализация
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.squeeze(0)

# Получаем вектор для нашей выжимки диалога и векторы для всей базы FAQ
query_emb = get_embedding(dialogue_summary, is_query=True)  # Векторизация dialogue_summary, is_query=True
faq_embs = torch.stack([get_embedding(p, is_query=False) for p in faq_passages])  # Векторизация всех элементов faq_passages, is_query=False (torch.stack)

# TODO: Посчитайте косинусное сходство (dot product) между query_emb и faq_embs
similarities = torch.matmul(faq_embs, query_emb)  # Используйте torch.matmul
best_idx = torch.argmax(similarities).item()  # Найдите индекс максимального значения

print("Лучший ответ из FAQ:", faq_passages[best_idx] if best_idx is not None else "Не найден")

Лучший ответ из FAQ: Если сломался телефон после обновления, рекомендуется сбросить устройство до заводских настроек.


### Автопроверка Модуля 3


In [15]:
assert query_emb is not None and len(query_emb.shape) == 1, "Эмбеддинг должен быть 1D тензором"
assert abs(torch.norm(query_emb).item() - 1.0) < 1e-4, "Эмбеддинг должен быть L2-нормализован"

# Хешируем индекс правильного ответа (вопрос про прошивку -> ответ про прошивку (индекс 1))
assert hashlib.md5(str(best_idx).encode()).hexdigest() == 'c4ca4238a0b923820dcc509a6f75849b', "Неверно определен наиболее релевантный документ"

print("Модуль 3 пройден успешно!")


Модуль 3 пройден успешно!


---
## Модуль 4: Генерация окончательного ответа пользователю (Instruct SLM)

В финальной стадии мы сгенерируем умный и вежливый ответ саппорта автоматически. Мы используем компактную, но очень мощную модель `Qwen/Qwen2.5-1.5B-Instruct` (она отлично поместится в 8 Гб GPU благодаря загрузке в 16-битном формате).

**Ваша задача:** Написать логику загрузки с `torch.bfloat16`, подготовить `chat_template` из словарей ролей, передав выявленную суть проблемы и найденный факт из БД, и сгенерировать финальный ответ.


In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name_gpt = "Qwen/Qwen2.5-1.5B-Instruct"

# TODO: Загрузите токенизатор и модель.
# ВНИМАНИЕ: Для модели укажите параметр torch_dtype=torch.bfloat16 (или float16), иначе при генерации может не хватить памяти GPU!
tokenizer_gpt = AutoTokenizer.from_pretrained(model_name_gpt, trust_remote_code=True)
model_gpt = AutoModelForCausalLM.from_pretrained(
    model_name_gpt, 
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)
if device == "cpu":
    model_gpt = model_gpt.to(device)

retrieved_doc = faq_passages[best_idx] if best_idx is not None else faq_passages[1]

# TODO: Составьте сообщения (список словарей) для Chat-формата:
# Обязательно укажите роль "system" (определите поведение саппорта) и "user" (передайте dialogue_summary и retrieved_doc)
messages = [
    {"role": "system", "content": "Ты — вежливый и профессиональный оператор технической поддержки. Отвечай кратко, по делу и доброжелательно. Используй информацию из контекста для ответа пользователю."},
    {"role": "user", "content": f"Проблема пользователя: {dialogue_summary}\n\nИнформация из базы знаний: {retrieved_doc}\n\nПожалуйста, дай ответ пользователю, объяснив, как решить проблему."}
]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 204.14it/s]


In [17]:
def generate_response(chat_messages):
    # TODO: Вызовите apply_chat_template для правильного форматирования промпта
    # Сгенерируйте ответ с помощью model_gpt.generate()
    # Обрежьте входной промпт, чтобы вернуть только финальный ответ системного ассистента!
    
    text = tokenizer_gpt.apply_chat_template(
        chat_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer_gpt(text, return_tensors="pt").to(model_gpt.device)
    
    with torch.no_grad():
        outputs = model_gpt.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer_gpt.eos_token_id
        )
    
    response = tokenizer_gpt.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

final_response = generate_response(messages)  # Замените на вызов функции generate_response(messages)
print("Сгенерированный ответ оператора:\n", final_response)

Сгенерированный ответ оператора:
 Добрый день! Вы знаете, какой способ действий лучше всего использовать при таких проблемах?

Можно попробовать сбросить вашу телефонную модель до заводских настроек, чтобы исправить возможные ошибки или проблемы, вызванные новой прошивкой. Это поможет восстановить все настройки телефона на первоначальные параметры, которые могут помочь вам снова начать пользоваться своим устройством без проблем.

Если вы не уверены, как это сделать, или если ситуация кажется слишком сложной, лучше обратиться за помощью к специалистам в сервисном центре, где они смогут проверить устройство более детально и предложить правильное решение.


### Автопроверка Модуля 4


In [18]:
assert final_response is not None and len(final_response) > 0, "Ответ должен быть непустой строкой"
assert type(messages) is list and len(messages) > 0, "Сообщения должны быть в виде списка словарей (чат-формат)"
assert "Контекст проблемы:" not in final_response, "В итоговом ответе не должно быть технической информации из промпта!"

print("✅ Итоговое задание (сквозной конвейер) полностью выполнено!")


✅ Итоговое задание (сквозной конвейер) полностью выполнено!
